In [ ]:
import pathlib
import random
import copy
import gc
import numpy as np
import torch
import torchvision
import quantus
import torchvision
from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from scipy.signal import hilbert
from sklearn.preprocessing import MinMaxScaler
from scipy.io import loadmat
from tqdm import tqdm

from pycircstat.tests import *
import statsmodels.multivariate.multivariate_ols as mv_ols
import statsmodels.api as sm 
import pandas as pd

# boilerplate

In [ ]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [2]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""

def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_shape_st = (60, 900)

In [ ]:
device

In [ ]:

def load_model(cfg, start_index=100, subject_index=2):
    save_path = cfg.dataset.data_directory
    file_path = os.path.join(save_path, f"subject_{subject_index}", f"model_checkpoint_pretrain_subject_index_{subject_index}_start_idx_{start_index}.pth") 
    trunk_net = TrunkNet(n_chans=input_shape_st[0], n_times=input_shape_st[1])
    head_net = HeadNet(64, 1)  # Assuming these are the correct dimensions
    model = S4PatchedFinalNet(64, trunk_net, head_net)
    
    weights = torch.load(file_path)
    model.load_state_dict(weights)
    #model.load_state_dict(full_checkpoint['model_state_dict'])
    model.eval()
    model.to(device)
    return model

In [ ]:
def plot_binned_averages2(ch_names, data, pred_label, degree_bins, axs, index_group=None):
    for idx, ch in enumerate(ch_names):
            if index_group is None:
                index_group = np.ones(len(data[ch]), dtype=bool)

            data_ch = data[ch][index_group]
            pred_label_ch = pred_label[index_group]

            
            df = pd.DataFrame(data={"pred":pred_label_ch, "cos":np.cos(data_ch), "sin":np.sin(data_ch)})
            X = df[['cos', 'sin']]
            y = df['pred'] 
            X = sm.add_constant(X) 
            est = sm.OLS(y, X).fit()

            coef_df = est.summary2().tables[1]
            const_p, cos_p, sin_p = coef_df["P>|t|"]   
            coeff1, coeff2, coeff3 = est.params 
            f_pvalue = est.f_pvalue


            def phasefit(x):
                return coeff1 + coeff2*np.cos(x) + coeff3*np.sin(x)
        
            def bestfit(x):
                return 0.2*(coeff1+np.cos(x))

            bars_pred = []
            bars_true = []
            for degree_bin in degree_bins:
                bin_mask = (np.rad2deg(data_ch) >= degree_bin) & (np.rad2deg(data_ch) < degree_bin + 36)
                bin_avg_pred = np.mean(pred_label_ch[bin_mask])
                #bin_avg_true = np.mean(labels_raw[bin_mask])

                #incidences.append(sum(bin_mask))
                bars_pred.append(bin_avg_pred)
                #bars_true.append(bin_avg_true)

            axs[idx // 4, idx % 4].bar(degree_bins, bars_pred, width=36, alpha=0.5, label='Predicted')
            axs[idx // 4, idx % 4].plot(degrees, bestfit(np.deg2rad(degrees)),  'g--', label='Phase fit')
            axs[idx // 4, idx % 4].plot(degrees, phasefit(np.deg2rad(degrees)), 'r', label='Best fit')
            #axs[idx // 4, idx % 4].bar(degree_bins, bars_true, width=36, alpha=0.5, label='True')
            phase_effec_str = ""
            if const_p < 0.05:
                phase_effec_str += f"p-val const={const_p:.2}\n"
            if cos_p < 0.05:
                phase_effec_str += f"p-val cos={cos_p:.2}\n"
            if sin_p < 0.05:
                phase_effec_str += f"p-val sin={sin_p:.2}\n"
            if f_pvalue < 0.05:
                phase_effec_str += f"p-val F={f_pvalue:.2}\n"
                axs[idx // 4, idx % 4].set_title(f"PHASE EFFECT Channel {ch}\n"+phase_effec_str, y=0.8, fontsize=13)
            else:
                axs[idx // 4, idx % 4].set_title(f"Channel {ch}\n"+phase_effec_str, y=0.8)
            axs[idx // 4, idx % 4].legend()
            axs[idx // 4, idx % 4].set_xlim([-270, 90])

In [ ]:
def plot_significant_phase_effects(ch_names, data, pred_label, degree_bins, axs, index_group=None):
    significant_channels = []
    for idx, ch in enumerate(ch_names):
        if index_group is not None:
            data_ch = data[ch][index_group]
            pred_label_ch = pred_label[index_group]
        else:
            data_ch = data[ch]
            pred_label_ch = pred_label

        df = pd.DataFrame(data={"pred": pred_label_ch, "cos": np.cos(data_ch), "sin": np.sin(data_ch)})
        X = df[['cos', 'sin']]
        y = df['pred']
        X = sm.add_constant(X)
        est = sm.OLS(y, X).fit()

        coeff1, coeff2, coeff3 = est.params
        f_pvalue = est.f_pvalue

        if f_pvalue < 0.05:
            fig, ax = plt.subplots()
            significant_channels.append(ch)
            def phasefit(x):
                return coeff1 + coeff2 * np.cos(x) + coeff3 * np.sin(x)

            def bestfit(x):
                return 0.2 * (coeff1 + np.cos(x))

            bars_pred = []
            for degree_bin in degree_bins:
                bin_mask = (np.rad2deg(data_ch) >= degree_bin) & (np.rad2deg(data_ch) < degree_bin + 36)
                bin_avg_pred = np.mean(pred_label_ch[bin_mask])
                bars_pred.append(bin_avg_pred)

            axs[idx // 4, idx % 4].bar(degree_bins, bars_pred, width=36, alpha=0.5, label='Predicted')
            axs[idx // 4, idx % 4].plot(degrees, bestfit(np.deg2rad(degrees)), 'g--', label='Phase fit')
            axs[idx // 4, idx % 4].plot(degrees, phasefit(np.deg2rad(degrees)), 'r', label='Best fit')
            axs[idx // 4, idx % 4].set_title(f"PHASE EFFECT Channel {ch}\np-val F={f_pvalue:.2}", y=0.8, fontsize=13)
            axs[idx // 4, idx % 4].legend()
            axs[idx // 4, idx % 4].set_xlim([-270, 90])
    return significant_channels


# for subject 2

## load data

In [ ]:
cfg = load_config()

subject_index = cfg.dataset.test_subject_indices[0]
cfg.dataset.subject_index = subject_index
cfg.exp_name = f"S4_S4EEGNet_ema_100_cal_py_{cfg.dataset.subject_index}"
cli_args = parse_args()
cfg = update_config(cfg, cli_args)
save_config(cfg)
all_epochs, labels_raw, _, _, _, _, _, ch_names = load_eeg_data(cfg)
all_epochs = all_epochs[150:]
labels_raw = labels_raw[150:]

correlate phase at TMS pulse with predicted labels and with actual raw_labels to check if correlation can be found

do this separately for each channel and each frequency band

recheck how they do it in phastimate paper.

In [ ]:
ch_names.remove("T7")
ch_names.remove("T8")
ch_names.remove("Iz")

In [ ]:
load_dir= "/home/marco/Documents/GitHub/tms_eeg_decoding/Pulse_phase_eval/results"
load_file = os.path.join(load_dir, "subj_2", "subj_2_freq_band_{}_ch_{}_900.mat")

freq_bands = ["theta", "alpha", "beta", "gamma"]
data_EEG = {k: {ch:[] for ch in ch_names} for k in freq_bands}
data_coeffs = {k: {ch:[] for ch in ch_names} for k in freq_bands}
for freq_band in freq_bands:
    for ch in ch_names:
        data_freq_ch = loadmat(load_file.format(freq_band, ch))
        data_EEG[freq_band][ch] = data_freq_ch["estphase"].squeeze()
        data_coeffs[freq_band][ch] = data_freq_ch["coeffs"].squeeze()

In [ ]:
data_freq_ch["coeffs"]

## for alpha frequency band

### phase amplitude scatter plot

presumably channels C3 or C4 will be the most interesting but we will see

In [ ]:
# predicted labels same for all channels and freq bands
pred_label = loadmat(load_file.format("alpha", "C4"))["y"]
pred_label = pred_label.squeeze()

In [ ]:

#fig, axs = plt.subplots(nrows=15, ncols=4, figsize=(20, 40))
#for idx,ch in enumerate(ch_names):
#    min_val = np.min(np.rad2deg(data["alpha"][ch]))
#    max_val = np.max(np.rad2deg(data["alpha"][ch]))
#    axs[idx//4, idx%4].scatter(np.rad2deg(data["alpha"][ch]), pred_label, c=uncertainties, alpha=0.5, label= f"min: {min_val:.2f}, max: {max_val:.2f}")
#    axs[idx//4, idx%4].set_title(ch, y=0.88)
#    axs[idx//4, idx%4].legend()
def plot_scatter_with_uncertainties(ch_names, data, pred_label):
    fig, axs = plt.subplots(nrows=15, ncols=4, figsize=(20, 40))
    for idx, ch in enumerate(ch_names):
        min_val = np.min(np.rad2deg(data[ch]))
        max_val = np.max(np.rad2deg(data[ch]))
        axs[idx // 4, idx % 4].scatter(np.rad2deg(data[ch]), pred_label,alpha=0.5, label=f"min: {min_val:.2f}, max: {max_val:.2f}")
        axs[idx // 4, idx % 4].set_title(ch, y=0.88)
        axs[idx // 4, idx % 4].legend()
    plt.show()

plot_scatter_with_uncertainties(ch_names, data_EEG["alpha"], pred_label)


In [ ]:
degrees = np.arange(-270, 90,1)

### circular linear model fit from matlab (old)

In [ ]:

#sinus_degrees = np.arange(-270, 90, 360)
#fig, axs = plt.subplots(nrows=15, ncols=4, figsize=(20, 40))
#for idx,ch in enumerate(ch_names):
#    bars_pred = []
##    bars_true = []
#   for degree_bin in degree_bins:
#       bin_mask = (np.rad2deg(data["alpha"][ch]) >= degree_bin) & (np.rad2deg(data["alpha"][ch]) < degree_bin + 36)
#        bin_avg_pred = np.mean(pred_label[bin_mask])
#        bin_avg_true = np.mean(labels_raw[bin_mask])
#        bars_pred.append(bin_avg_pred)
#        bars_true.append(bin_avg_true)
#    axs[idx//4, idx%4].bar(degree_bins, bars_pred, width=36)
#    axs[idx//4, idx%4].set_title(ch, y=0.88)
#    axs[idx//4, idx%4].legend()
#    axs[idx//4, idx%4].set_xlim([-270, 90])
fig, axs = plt.subplots(nrows=15, ncols=4, figsize=(20, 35))
degree_bins = np.arange(-270,90, 36)
degrees = np.arange(-270, 90,1)
def plot_binned_averages(ch_names, data, coeffs, pred_label, labels_raw, degree_bins, axs):
    for idx, ch in enumerate(ch_names):
        bars_pred = []
        bars_true = []
        coeff1, coeff2, coeff3 = coeffs[ch]

        def phasefit(x):
            return coeff1 + coeff2*np.cos(x) + coeff3*np.sin(x)
        
        def bestfit(x):
            return 0.2*(coeff1+np.cos(x))
        
        #model = mv_ols(pred_label, np.array([np.cos(data[ch]), np.sin(data[ch])]).T)
     
        #incidences = []
        for degree_bin in degree_bins:
            bin_mask = (np.rad2deg(data[ch]) >= degree_bin) & (np.rad2deg(data[ch]) < degree_bin + 36)
            bin_avg_pred = np.mean(pred_label[bin_mask])
            bin_avg_true = np.mean(labels_raw[bin_mask])

            #incidences.append(sum(bin_mask))
            bars_pred.append(bin_avg_pred)
            bars_true.append(bin_avg_true)
        
        #pval,z = rayleigh(np.array(degree_bins),np.array(incidences))

        axs[idx // 4, idx % 4].bar(degree_bins, bars_pred, width=36, alpha=0.5, label='Predicted')
        axs[idx // 4, idx % 4].plot(degrees, bestfit(np.deg2rad(degrees)),  'g--', label='Phase fit')
        axs[idx // 4, idx % 4].plot(degrees, phasefit(np.deg2rad(degrees)), 'r', label='Best fit')
        #axs[idx // 4, idx % 4].bar(degree_bins, bars_true, width=36, alpha=0.5, label='True')
        axs[idx // 4, idx % 4].set_title(f"Channel {ch}", y=0.88)
        axs[idx // 4, idx % 4].legend()
        axs[idx // 4, idx % 4].set_xlim([-270, 90])


plot_binned_averages(ch_names, data_EEG["alpha"], data_coeffs["alpha"], pred_label, labels_raw, degree_bins, axs)

### coeff computation with statsmodels (new)

In [ ]:
fig, axs = plt.subplots(nrows=15, ncols=4, figsize=(20, 35))
degree_bins = np.arange(-270,90, 36)
degrees = np.arange(-270, 90,1)
plot_binned_averages2(ch_names, data_EEG["alpha"],pred_label, degree_bins, axs)

In [ ]:
plot_significant_phase_effects(ch_names, data_EEG["alpha"], pred_label, degree_bins, axs)

interesting to notice: Most important Channel C4 is exact mirror of channel C3. Also note that channel PO8 which as also one of the most important seems very similar to C3 in its phase

## beta phase

In [ ]:
fig, axs = plt.subplots(nrows=15, ncols=4, figsize=(20, 40))
degree_bins = np.arange(-270,90, 36)
plot_binned_averages2(ch_names, data_EEG["beta"], pred_label, degree_bins, axs)

## gamma

In [ ]:
fig, axs = plt.subplots(nrows=15, ncols=4, figsize=(20, 40))
degree_bins = np.arange(-270,90, 36)
plot_binned_averages2(ch_names, data_EEG["gamma"], pred_label, degree_bins, axs)

## theta

In [ ]:
fig, axs = plt.subplots(nrows=15, ncols=4, figsize=(20, 40))
degree_bins = np.arange(-270,90, 36)
plot_binned_averages2(ch_names, data_EEG["theta"], pred_label, degree_bins, axs)

# development of effect over trials for freq-band alpha

In [ ]:
#for all trials in all_epochs, get boolean index groups such every 100 subsequent trials are grouped together
index_groups_all= []
for i in range(0,len(all_epochs), 100):
    if len(all_epochs) > i+100:
        index_group = np.zeros(len(all_epochs), dtype=bool)
        index_group[i:i+100] = True
        index_groups_all.append(index_group)
    else:
        index_group = np.zeros(len(all_epochs), dtype=bool)
        index_group[i:] = True
        index_groups_all.append(index_group)


In [ ]:
fig, axs = plt.subplots(nrows=15, ncols=4, figsize=(20, 45))
fig.tight_layout()
degree_bins = np.arange(-270,90, 36)
degrees = np.arange(-270, 90,1)
def plot_binned_averages2(ch_names, data, pred_label, degree_bins, axs, index_group=None):
    for idx, ch in enumerate(ch_names):
            if index_group is not None:
                 data_ch = data[ch][index_group]
                 pred_label_ch = pred_label[index_group]
            else:
                data_ch = data[ch]
                pred_label_ch = pred_label
            
            df = pd.DataFrame(data={"pred":pred_label_ch, "cos":np.cos(data_ch), "sin":np.sin(data_ch)})
            X = df[['cos', 'sin']]
            y = df['pred'] 
            X = sm.add_constant(X) 
            est = sm.OLS(y, X).fit()
           
            coeff1, coeff2, coeff3 = est.params 
            f_pvalue = est.f_pvalue

            def phasefit(x):
                return coeff1 + coeff2*np.cos(x) + coeff3*np.sin(x)
        
            def bestfit(x):
                return 0.2*(coeff1+np.cos(x))
            
            coef_df = est.summary2().tables[1]
            const_p, cos_p, sin_p = coef_df["P>|t|"]

            bars_pred = []
            bars_true = []
            for degree_bin in degree_bins:
                bin_mask = (np.rad2deg(data_ch) >= degree_bin) & (np.rad2deg(data_ch) < degree_bin + 36)
                bin_avg_pred = np.mean(pred_label_ch[bin_mask])
                #bin_avg_true = np.mean(labels_raw[bin_mask])

                #incidences.append(sum(bin_mask))
                bars_pred.append(bin_avg_pred)
                #bars_true.append(bin_avg_true)

            axs[idx // 4, idx % 4].bar(degree_bins, bars_pred, width=36, alpha=0.5, label='Predicted')
            axs[idx // 4, idx % 4].plot(degrees, bestfit(np.deg2rad(degrees)),  'g--', label='Phase fit')
            axs[idx // 4, idx % 4].plot(degrees, phasefit(np.deg2rad(degrees)), 'r', label='Best fit')
            #axs[idx // 4, idx % 4].bar(degree_bins, bars_true, width=36, alpha=0.5, label='True')
            phase_effec_str = ""
            if const_p < 0.05:
                phase_effec_str += f"p-val const={const_p:.2}\n"
            if cos_p < 0.05:
                phase_effec_str += f"p-val cos={cos_p:.2}\n"
            if sin_p < 0.05:
                phase_effec_str += f"p-val sin={sin_p:.2}\n"
            if f_pvalue < 0.05:
                phase_effec_str += f"p-val F={f_pvalue:.2}\n"
                axs[idx // 4, idx % 4].set_title(f"PHASE EFFECT Channel {ch}\n"+phase_effec_str, y=0.8, fontsize=13)
            else:
                axs[idx // 4, idx % 4].set_title(f"Channel {ch}\n"+phase_effec_str, y=0.8)
            axs[idx // 4, idx % 4].legend()
            axs[idx // 4, idx % 4].set_xlim([-270, 90])

plot_binned_averages2(ch_names, data_EEG["alpha"], pred_label, degree_bins, axs)

In [ ]:
# coefficient estimated by matlab ln and statsmodels are very similar as desired
for idx, ch in enumerate(ch_names):
    bars_pred = []
    bars_true = []
    df = pd.DataFrame(data={"pred":pred_label, "cos":np.cos(data_EEG["alpha"][ch]), "sin":np.sin(data_EEG["alpha"][ch])})
    X = df[['cos', 'sin']]
    y = df['pred'] 
    X = sm.add_constant(X) 
    est = sm.OLS(y, X).fit()
    coeff1, coeff2, coeff3 = est.params 
    #print(f"{est.params}")
    #print(data_coeffs["alpha"][ch])
    #print("\n")

    #model = mv_ols(pred_label[:,np.newaxis], np.array([np.cos(data_EEG["alpha"][ch]), np.sin(data_EEG["alpha"][ch])]).T)
    

## trials 0-100

In [ ]:
fig, axs = plt.subplots(nrows=15, ncols=4, figsize=(20, 40))
fig.tight_layout()
plot_binned_averages2(ch_names, data_EEG["alpha"], pred_label, degree_bins, axs, index_group=index_groups_all[0])

## trials 100-200

In [ ]:
fig, axs = plt.subplots(nrows=15, ncols=4, figsize=(20, 40))
fig.tight_layout()
plot_binned_averages2(ch_names, data_EEG["alpha"], pred_label, degree_bins, axs, index_group=index_groups_all[1])

## trials 200-300

In [ ]:
fig, axs = plt.subplots(nrows=15, ncols=4, figsize=(20, 40))
fig.tight_layout()
plot_binned_averages2(ch_names, data_EEG["alpha"], pred_label, degree_bins, axs, index_group=index_groups_all[2])

## trials 300-400

In [ ]:
fig, axs = plt.subplots(nrows=15, ncols=4, figsize=(20, 40))
fig.tight_layout()
plot_binned_averages2(ch_names, data_EEG["alpha"], pred_label, degree_bins, axs, index_group=index_groups_all[3])

## trials 400-500

In [ ]:
fig, axs = plt.subplots(nrows=15, ncols=4, figsize=(20, 40))
fig.tight_layout()
plot_binned_averages2(ch_names, data_EEG["alpha"], pred_label, degree_bins, axs, index_group=index_groups_all[4])

# development of effect over trials for freq-band beta

## trials 0-100

In [ ]:
fig, axs = plt.subplots(nrows=15, ncols=4, figsize=(20, 40))
fig.tight_layout()
plot_binned_averages2(ch_names, data_EEG["beta"], pred_label, degree_bins, axs, index_group=index_groups_all[0])

## trials 100-200

In [ ]:
fig, axs = plt.subplots(nrows=15, ncols=4, figsize=(20, 40))
fig.tight_layout()
plot_binned_averages2(ch_names, data_EEG["beta"], pred_label, degree_bins, axs, index_group=index_groups_all[1])

## trials 200-300

In [ ]:
fig, axs = plt.subplots(nrows=15, ncols=4, figsize=(20, 40))
fig.tight_layout()
plot_binned_averages2(ch_names, data_EEG["beta"], pred_label, degree_bins, axs, index_group=index_groups_all[3])

## trials 300-400

In [ ]:
fig, axs = plt.subplots(nrows=15, ncols=4, figsize=(20, 40))
fig.tight_layout()
plot_binned_averages2(ch_names, data_EEG["beta"], pred_label, degree_bins, axs, index_group=index_groups_all[4])

## trials 400-500

In [ ]:
fig, axs = plt.subplots(nrows=15, ncols=4, figsize=(20, 40))
fig.tight_layout()
plot_binned_averages2(ch_names, data_EEG["beta"], pred_label, degree_bins, axs, index_group=index_groups_all[5])

# development of effect over trials for freq-band gamma

## index 0-100

In [ ]:
fig, axs = plt.subplots(nrows=15, ncols=4, figsize=(20, 40))
fig.tight_layout()
plot_binned_averages2(ch_names, data_EEG["gamma"], pred_label, degree_bins, axs, index_group=index_groups_all[0])

## index 100-200

In [ ]:
fig, axs = plt.subplots(nrows=15, ncols=4, figsize=(20, 40))
fig.tight_layout()
plot_binned_averages2(ch_names, data_EEG["gamma"], pred_label, degree_bins, axs, index_group=index_groups_all[1])

## index 200-300

In [ ]:
fig, axs = plt.subplots(nrows=15, ncols=4, figsize=(20, 40))
fig.tight_layout()
plot_binned_averages2(ch_names, data_EEG["gamma"], pred_label, degree_bins, axs, index_group=index_groups_all[2])

## index 300-400

In [ ]:
fig, axs = plt.subplots(nrows=15, ncols=4, figsize=(20, 40))
fig.tight_layout()
plot_binned_averages2(ch_names, data_EEG["gamma"], pred_label, degree_bins, axs, index_group=index_groups_all[3])

## index 400-500

In [ ]:
fig, axs = plt.subplots(nrows=15, ncols=4, figsize=(20, 40))
fig.tight_layout()
plot_binned_averages2(ch_names, data_EEG["gamma"], pred_label, degree_bins, axs, index_group=index_groups_all[4])

## index 500-600

In [ ]:
fig, axs = plt.subplots(nrows=15, ncols=4, figsize=(20, 40))
fig.tight_layout()
plot_binned_averages2(ch_names, data_EEG["gamma"], pred_label, degree_bins, axs, index_group=index_groups_all[5])

ToDo, check phase effects across subjects, only plot and collect signifcant channels